In [2]:
!pip install shap

   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   ------------------- -------------------- 262.1/549.1 kB ? eta -:--:--
   ---------------------------------------- 549.1/549.1 kB 1.4 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]



In [3]:
import pandas as pd
import joblib
import shap
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------------
# LOAD ARTIFACTS (ONCE AT SERVER START)
# -------------------------------
model = joblib.load("model.pkl")
tfidf = joblib.load("tfidf.pkl")
columns = joblib.load("columns.pkl")

explainer = shap.TreeExplainer(model)

# -------------------------------
# HELPER → BUILD FEATURE ROW
# -------------------------------
def build_feature_row(startup_data, investor_data):

    df = pd.DataFrame([{
        "startup_industry": startup_data["industry"],
        "startup_stage": startup_data["stage"],
        "funding_required_lakhs": startup_data["funding_required"],

        "investor_pref_industry": investor_data["preferred_industry"],
        "investor_pref_stage": investor_data["preferred_stage"],
        "investor_ticket_min_lakhs": investor_data["ticket_min"],
        "investor_ticket_max_lakhs": investor_data["ticket_max"],

        "startup_desc": startup_data["description"],
        "investor_desc": investor_data["description"]
    }])

    # Idea similarity
    s_vec = tfidf.transform(df["startup_desc"])
    i_vec = tfidf.transform(df["investor_desc"])
    similarity = cosine_similarity(s_vec, i_vec)[0][0]

    df["idea_similarity"] = similarity

    # Funding fit
    df["funding_fit"] = (
        (df["funding_required_lakhs"] >= df["investor_ticket_min_lakhs"]) &
        (df["funding_required_lakhs"] <= df["investor_ticket_max_lakhs"])
    ).astype(int)

    df = df.drop(["startup_desc", "investor_desc"], axis=1)

    df_encoded = pd.get_dummies(df, drop_first=True)

    df_aligned = df_encoded.reindex(columns=columns, fill_value=0)

    return df_aligned

# -------------------------------
# EXPLANATION ENGINE ⭐⭐⭐⭐⭐
# -------------------------------
def explain_match(startup_data, investor_data):

    X_input = build_feature_row(startup_data, investor_data)

    prediction = model.predict(X_input)[0]
    prediction = float(max(0, min(100, prediction)))

    shap_vals = explainer.shap_values(X_input)[0]

    feature_contrib = dict(zip(X_input.columns, shap_vals))

    # Aggregate into human concepts
    contributions = {
        "Industry Fit": 0,
        "Stage Fit": 0,
        "Funding Fit": 0,
        "Idea Similarity": 0
    }

    for feature, value in feature_contrib.items():

        if "industry" in feature.lower():
            contributions["Industry Fit"] += value

        elif "stage" in feature.lower():
            contributions["Stage Fit"] += value

        elif "funding_fit" in feature.lower():
            contributions["Funding Fit"] += value

        elif "idea_similarity" in feature.lower():
            contributions["Idea Similarity"] += value

    # Clean rounding for UI
    for k in contributions:
        contributions[k] = round(float(contributions[k]), 2)

    return {
        "match_score": round(prediction, 2),
        "contributions": contributions
    }

# -------------------------------
# EXAMPLE TEST ⭐⭐⭐⭐⭐
# -------------------------------
startup_example = {
    "industry": "FinTech",
    "stage": "Seed",
    "funding_required": 80,
    "description": "AI-based fraud detection for digital payments"
}

investor_example = {
    "preferred_industry": "FinTech",
    "preferred_stage": "Seed",
    "ticket_min": 50,
    "ticket_max": 200,
    "description": "Investing in FinTech and AI-driven startups"
}

result = explain_match(startup_example, investor_example)

print(result)

{'match_score': 68.88, 'contributions': {'Industry Fit': -0.61, 'Stage Fit': -1.74, 'Funding Fit': 8.28, 'Idea Similarity': 28.29}}
